# Reading findings

`report.findings` is a list of dictionaries. Each entry describes a key,
dependency, association, or data-quality issue. Start with `claim` for a
readable summary, then inspect the columns and measurements behind it.

| Field | Contents |
|---|---|
| `id` | Stable identifier for the same category, type, columns, and specification |
| `category` | Kind of result, such as `uniqueness` or `dependencies` |
| `type` | Subtype, such as a `functional` or `predictive` dependency |
| `subject` | Columns involved; dependencies also name their inputs and target |
| `specification` | Formula, mapping, conditions, outcome, or comparison definition |
| `claim` | Text summary of the result |
| `evidence` | Methods, evaluated rows, validation, and measured or fitted values |
| `interestingness` | Score used to order findings in the report |
| `interpretation` | Suggested action or meaning, when available |

The example adds a concentrated sales channel and mixed-format invoice
numbers to an orders table.


In [ ]:
import json
import random

import polars as pl

import onceover

rng = random.Random(7)
n = 1500

region_id = [rng.randrange(3) for _ in range(n)]
region_name = [["emea", "amer", "apac"][r] for r in region_id]
order_id = list(range(n))
external_ref = [f"R{i:05}" for i in range(n)]
external_ref[-1] = external_ref[0]
external_ref[-2] = external_ref[1]
quantity = [rng.randrange(1, 11) for _ in range(n)]
double_quantity = [2 * q for q in quantity]
channel = ["web" if rng.random() < 0.99 else "app" for _ in range(n)]
invoice_no = [f"INV-{i:04d}" if i % 2 == 0 else f"{i:04d}" for i in range(n)]

df = pl.DataFrame({
    "order_id": order_id,
    "external_ref": external_ref,
    "region_id": region_id,
    "region_name": region_name,
    "quantity": quantity,
    "double_quantity": double_quantity,
    "channel": channel,
    "invoice_no": invoice_no,
})

report = onceover.profile(df)
print(report.overview["headline"])

## Categories

The category describes the result. To identify the method that produced
it, read `evidence[*].method.detector`.

| Category | Type | Reports |
|---|---|---|
| `missingness` | — | Missing-value counts, including entirely null columns |
| `concentration` | — | Constant columns or a distribution dominated by a few values |
| `concentration` | `modal_share`, `spike_cells` | Views' within-cluster concentration and low-density cells |
| `column_roles` | — | Suggested interpretations such as category codes or identifiers |
| `uniqueness` | — | Exact or nearly unique columns and column combinations |
| `duplicates` | — | Repeated rows under the specified equality rule |
| `formats` | — | Mixed string formats |
| `dependencies` | `functional` | Value mappings and affine identities, with violations or tolerances |
| `dependencies` | `predictive` | Prediction of a target from named inputs, including local linear models |
| `conditional_rules` | `if_then` | Conditions, an outcome, and how often that outcome holds |
| `associations` | `statistical` | Numeric dependence measures or views' shared row partitions |
| `gaps` | — | Missing dates or values relative to a stated sequence |
| `intervals` | — | Irregular temporal spacing |
| `ordering` | — | Sortedness in the existing row order |
| `changes` | `residual_level` | Dated residual shifts, or views' differences between unordered clusters |

Views is opt-in. Its cluster findings do not identify temporal change
points; see the [views chapter](../methodology/view_clusters.ipynb).

Count findings by category to see what this report contains:


In [ ]:
from collections import Counter

print(Counter((f["category"], f.get("type")) for f in report.findings))

## Methods and scores

The following result predicts `region_id` from `region_name`. Both the
boosted model (`gbm`) and the decision table found this relationship, so
the finding contains evidence from both.

The boosted model reports accuracy on a holdout. The decision table
reports cross-validated accuracy. Inspect `method.detector` and
`validation.method` before comparing scores: the models and evaluated
rows differ.


In [ ]:
merged = next(f for f in report.findings
              if f["category"] == "dependencies"
              and len(f["evidence"]) > 1)
print(merged["claim"])
for ev in merged["evidence"]:
    m = ev["method"]
    print(f"  {m['detector']:16s} basis={ev['basis']:9s} "
          f"validation={ev['validation']['method']}")

Both methods report perfect accuracy in this fixture. That result is
expected because each region name maps to exactly one region code. The
functional-dependency findings check the mapping directly and report its
violation count.

Conditional rules also include discovery and evaluation results. In the
next example, PRISM finds that `double_quantity <= 2` implies `quantity = 1`.
The rule evaluator then counts matches on its diagnostic rows:


In [ ]:
rule = next(f for f in report.findings if f["category"] == "conditional_rules")
print(rule["claim"])
for ev in rule["evidence"]:
    print(f"  {ev['method']['detector']:16s} basis={ev['basis']:9s} "
          f"validation={ev['validation']['method']}")
print("  evaluated metrics:", {k: v for k, v in rule["evidence"][-1]["metrics"].items()
                                if k in ("conditional_rate", "baseline_rate", "lift", "joint_count")})

The rule holds in 38 of 38 matching diagnostic rows. Its baseline rate is
11.80%, so the reported lift is approximately 8.47. These are the
`rule_evaluator` metrics. PRISM's construction score describes the search
that produced the rule and should not be used as a replacement for those
counts.

## Finding identifiers

IDs are derived from the category, type, subject, and specification. Use
them to match the same finding across reports generated with the same
schema. They do not encode rank or detector order.

## Ranking

`interestingness` is a priority score. It combines evidence quality,
effect strength, specificity, and simplicity:

$$\text{interestingness} = 0.45 \cdot \text{quality} + 0.35 \cdot
\text{strength} + 0.10 \cdot \text{specificity} + 0.10 \cdot \text{simplicity}$$

- **Quality:** full-data direct comparisons score 1.0; holdouts,
  cross-validation, and diagnostic partitions score 0.8. Other evidence
  uses its basis: measured 0.85, estimated 0.6, inferred 0.4.
- **Strength:** the metric depends on the category. Examples include the
  proportion of rows satisfying a functional dependency, predictive
  performance, and a rule's improvement over its baseline.
- **Specificity:** a specified form, such as an affine formula, scores 1.0;
  an unspecified form scores 0.5.
- **Simplicity:** fewer inputs and conditions score higher, using
  $1/(1 + (\text{inputs} + \text{conditions})/8)$.

The quickstart's exact dependency scores 0.9889: its quality, strength,
and specificity each score 1.0, while its one input gives simplicity 0.889.
A score near one is not a probability of correctness.

## Report size

The selector sorts candidates and applies these limits:

| Setting | Default | Limit |
|---|---:|---|
| `max_findings` | 100 | Total findings |
| `max_rules` | 20 | Conditional rules |
| `rules_per_outcome` | 2 | Rules for the same outcome, such as `shipping = "express"` |
| `predictive_findings_per_target` | 3 | Predictive findings for one target |

Python exposes `max_findings` and `max_rules`. Rust's `SelectionPolicy`
exposes all four. `report.data["selection"]` shows how many candidates and
distinct findings were considered and how many were included.

## Related findings

`report.groups` connects findings that share columns. For example,
`quantity` and `double_quantity` appear together with the findings about
their relationship. Before removing a column as redundant, check whether
the other columns determine it and whether the mapping has exceptions.

## Filter findings

Select the categories relevant to your task:


In [ ]:
structure = [f for f in report.findings
             if f["category"] in ("uniqueness", "duplicates", "dependencies",
                                  "conditional_rules")]
print(len(structure), "structural findings")
for f in structure[:5]:
    print(f"  [{f['category']}/{f.get('type', '-')}] {f['claim'][:70]}")

To inspect a particular column, filter on `subject.columns`:


In [ ]:
for f in report.findings:
    if "quantity" in f["subject"]["columns"]:
        print(f"[{f['category']}/{f.get('type', '-')}] {f['claim'][:80]}")

To select findings with measurements over the full table, inspect each
evidence record's `basis` and `scope.sampling`:


In [ ]:
measured = [f for f in report.findings
              if any(ev["basis"] == "measured" and ev["scope"]["sampling"] == "full"
                     for ev in f["evidence"])]
print(len(measured), "statements measured on full data")

For a model result, also inspect `validation.method`. Cross-validation,
a holdout, and a diagnostic partition use different row sets and can
participate in model or finding selection. The
[evidence chapter](../methodology/evidence.ipynb) walks through an example.

## Acting on a finding

`interpretation` suggests possible actions. For example, a duplicate-row
finding may suggest checking for repeated imports. Confirm the intended
meaning of the rows before deleting them. Similarly, verify a column's
intended key or code semantics before changing a join or model.

For use on new data, test predictive results on observations that were not
used to fit or select the model. A diagnostic-partition rate describes the
recorded rows and may reuse discovery data.

If an expected finding is absent, inspect coverage. It lists skipped
passes and search limits; an absent finding alone does not establish that
the columns are independent.

## Select detectors

`onceover.available_detectors()` lists detector IDs, output categories,
and required detectors. Pass IDs in `detectors=[...]` to select which
findings are requested:


In [ ]:
for d in onceover.available_detectors()[:4]:
    print(f"{d['id']:24s} categories={d['categories']} requires={d['requires']}")

subset = onceover.profile(df, detectors=["keys", "functional_dependencies"])
print(subset.overview["headline"])

Some selected detectors need another detector's output. Decision tables,
for example, use target profiles from `gbm`. The required pass runs to
supply that data, but contributes findings only if it was also selected.
Coverage identifies these required passes and the passes that were not
enabled.


## Findings as a DataFrame

`report.findings_frame()` returns the ledger as a Polars DataFrame:
one row per finding with `id`, `category`, `type`, `columns`, `inputs`,
`target`, `interestingness`, `claim`, `n_evidence`, and the primary
evidence record's `basis`, in report order. Sorting and filtering the
frame is the programmatic counterpart to reading the Markdown report,
which prints each finding's evidence directly under its claim.


In [ ]:
frame = report.findings_frame()
print(frame.height, "findings")
print(frame.select(["category", "interestingness", "claim"]).head(3))